# Task 4: Distributed Computing

**Objective:** Analyze and demonstrate distributed computing concepts in PySpark including caching, persistence, repartitioning, Spark configuration monitoring, and resource utilization metrics as required in the assignment.

This notebook focuses on distributed computing aspects including caching strategies, persistence levels, partition management, Spark UI metrics, and resource monitoring for the flight delay prediction workload.

## Methodology

1. Load and prepare the dataset (reusing steps from Tasks 1-2)
2. Demonstrate caching and persistence techniques
3. Monitor and display Spark configuration
4. Analyze partitioning strategies and repartitioning effects
5. Measure and record resource utilization metrics
6. Provide guidance for Spark UI screenshots
7. Provide interpretation notes for academic report

In [ ]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import StorageLevel
import time

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("7006SCN-Task4-DistributedComputing") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Set log level
spark.sparkContext.setLogLevel("WARN")

# Load and prepare dataset (same as previous tasks)
DATA_PATH = "data/On_Time_Reporting_Carrier_On_Time_Performance_2024_1.csv"
print("Loading and preparing dataset...")

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "") \
    .option("treatEmptyValuesAsNulls", "true") \
    .csv(DATA_PATH)

# Apply data preparation steps
df_clean = df.dropDuplicates()

# Handle missing values
numerical_cols = [field.name for field in df_clean.schema.fields if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType))]
string_cols = [field.name for field in df_clean.schema.fields if isinstance(field.dataType, StringType)]

df_filled = df_clean
for col_name in numerical_cols:
    if col_name in df_clean.columns:
        median_val = df_clean.approxQuantile(col_name, [0.5], 0.25)[0]
        if median_val is not None:
            df_filled = df_filled.fillna({col_name: median_val})
        else:
            df_filled = df_filled.fillna({col_name: 0})

for col_name in string_cols:
    if col_name in df_clean.columns:
        df_filled = df_filled.fillna({col_name: "Unknown"})

# Remove cancelled and diverted flights
df_filtered = df_filled.filter((col("Cancelled") == 0) & (col("Diverted") == 0))

# Remove leakage variables and identifiers
leakage_vars = ["ArrDelay", "ArrDelayMinutes", "ArrivalDelayGroups", "ArrTime", "ArrTimeBlk"]
identifier_cols = ["Tail_Number", "Flight_Number_Reporting_Airline", "FlightDate"]
cols_to_remove = [c for c in leakage_vars + identifier_cols if c in df_filtered.columns]
df_clean = df_filtered.drop(*cols_to_remove)

# Select features
numeric_features = ["Month", "DayOfWeek", "CRSDepTime", "DepDelay", "TaxiOut", "Distance", "AirTime"]
categorical_features = ["Reporting_Airline", "Origin", "Dest"]
label_column = "ArrDel15"
required_columns = numeric_features + categorical_features + [label_column]
df_selected = df_clean.select(required_columns)

# Apply sampling
df_processed = df_selected.sampleBy(
    label_column,
    fractions={0: 0.15, 1: 0.15},
    seed=42
)

print(f"Dataset prepared: {df_processed.count():,} rows, {len(df_processed.columns)} columns")

## Distributed Computing Analysis

Demonstrate and measure distributed computing concepts as required.

In [ ]:
# ============================================================================
# CACHING AND PERSISTENCE
# ============================================================================

print("\n=== CACHING AND PERSISTENCE ANALYSIS ===")

# Initial state - not cached
print(f"Initial cache status: {df_processed.is_cached}")

# Apply caching
print("Applying cache()...")
start_time = time.time()
df_cached = df_processed.cache()
# Trigger caching by performing an action
cache_count = df_cached.count()
cache_time = time.time() - start_time
print(f"Cache status after cache(): {df_cached.is_cached}")
print(f"Time to cache and count: {cache_time:.4f} seconds")
print(f"Cached count: {cache_count:,}")

# Apply persistence with different storage levels
print("\nApplying persist() with different storage levels...")

storage_levels = [
    ("MEMORY_ONLY", StorageLevel.MEMORY_ONLY),
    ("MEMORY_AND_DISK", StorageLevel.MEMORY_AND_DISK),
    ("DISK_ONLY", StorageLevel.DISK_ONLY),
    ("MEMORY_ONLY_SER", StorageLevel.MEMORY_ONLY_SER),
    ("MEMORY_AND_DISK_SER", StorageLevel.MEMORY_AND_DISK_SER)
]

persistence_results = []

for level_name, level in storage_levels:
    start_time = time.time()
    df_persisted = df_processed.persist(level)
    persist_count = df_persisted.count()
    persist_time = time.time() - start_time
    
    persistence_results.append({
        "level": level_name,
        "is_persisted": df_persisted.is_cached,  # Note: is_cached returns True for both cache and persist
        "persist_time": persist_time,
        "count": persist_count
    })
    
    print(f"{level_name}: {persist_time:.4f} seconds, count: {persist_count:,}")

# Show storage level information
print(f"\nStorage level of cached data: {df_cached.getStorageLevel()}")

# Unpersist when done
df_cached.unpersist()
for level_name, level in storage_levels:
    df_processed.unpersist()
print("All data unpersisted")

In [ ]:
# ============================================================================
# SPARK CONFIGURATION MONITORING
# ============================================================================

print("\n=== SPARK CONFIGURATION ANALYSIS ===")

# Get and display Spark configuration
conf = spark.sparkContext.getConf()
conf_dict = dict(conf.getAll())

print("Key Spark Configuration Parameters:")
print(f"Application Name: {conf.get('spark.app.name', 'Not set')}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Master: {spark.sparkContext.master}")
print(f"Deploy Mode: {conf.get('spark.submit.deployMode', 'Not set')}")
print(f"Executor Instances: {conf.get('spark.executor.instances', 'Dynamic (default)')}")
print(f"Executor Memory: {conf.get('spark.executor.memory', '1g (default)')}")
print(f"Executor Cores: {conf.get('spark.executor.cores', '1 (default)')}")
print(f"Driver Memory: {conf.get('spark.driver.memory', '1g (default)')}")
print(f"Driver Cores: {conf.get('spark.driver.cores', '1 (default)')}")
print(f"Serializer: {conf.get('spark.serializer', 'org.apache.spark.serializer.JavaSerializer (default)')}")

# Adaptive query execution settings
print("\nAdaptive Query Execution Settings:")
print(f"spark.sql.adaptive.enabled: {conf.get('spark.sql.adaptive.enabled', 'false')}")
print(f"spark.sql.adaptive.coalescePartitions.enabled: {conf.get('spark.sql.adaptive.coalescePartitions.enabled', 'false')}")
print(f"spark.sql.adaptive.skewJoin.enabled: {conf.get('spark.sql.adaptive.skewJoin.enabled', 'false')}")
print(f"spark.sql.adaptive.localShuffleReader.enabled: {conf.get('spark.sql.adaptive.localShuffleReader.enabled', 'false')}")

# Show all configuration properties (first 20 for brevity)
print(f"\nTotal configuration properties: {len(conf_dict)}")
print("First 20 configuration properties:")
for i, (key, value) in enumerate(list(conf_dict.items())[:20]):
    print(f"  {key}: {value}")
if len(conf_dict) > 20:
    print(f"  ... and {len(conf_dict) - 20} more properties")

In [ ]:
# ============================================================================
# PARTITIONING ANALYSIS
# ============================================================================

print("\n=== PARTITIONING ANALYSIS ===")

# Initial partitioning
initial_partitions = df_processed.rdd.getNumPartitions()
print(f"Initial number of partitions: {initial_partitions}")

# Analyze partition sizes
partition_sizes = df_processed.rdd.glom().map(len).collect()
if partition_sizes:
    print(f"Partition size statistics:")
    print(f"  Minimum: {min(partition_sizes)} records")
    print(f"  Maximum: {max(partition_sizes)} records")
    print(f"  Average: {sum(partition_sizes)/len(partition_sizes):.0f} records")
    print(f"  Standard deviation: {((sum((x - sum(partition_sizes)/len(partition_sizes))**2 for x in partition_sizes) / len(partition_sizes))**0.5):.0f} records")

# Demonstrate repartitioning
print("\nRepartitioning to 200 partitions as required...")
start_time = time.time()
df_repartitioned = df_processed.repartition(200)
repartition_time = time.time() - start_time
repartitioned_partitions = df_repartitioned.rdd.getNumPartitions()
print(f"Repartitioning time: {repartition_time:.4f} seconds")
print(f"Number of partitions after repartitioning: {repartitioned_partitions}")

# Analyze repartitioned partition sizes
repartitioned_sizes = df_repartitioned.rdd.glom().map(len).collect()
if repartitioned_sizes:
    print(f"Repartitioned partition size statistics:")
    print(f"  Minimum: {min(repartitioned_sizes)} records")
    print(f"  Maximum: {max(repartitioned_sizes)} records")
    print(f"  Average: {sum(repartitioned_sizes)/len(repartitioned_sizes):.0f} records")

# Demonstrate coalescing (for decreasing partitions)
print("\nCoalescing to 50 partitions...")
start_time = time.time()
df_coalesced = df_repartitioned.coalesce(50)
coalesce_time = time.time() - start_time
coalesced_partitions = df_coalesced.rdd.getNumPartitions()
print(f"Coalescing time: {coalesce_time:.4f} seconds")
print(f"Number of partitions after coalescing: {coalesced_partitions}")

# Show final partitioning for modeling
print(f"\nFinal dataset partitioning for modeling: {df_processed.rdd.getNumPartitions()} partitions")

In [ ]:
# ============================================================================
# RESOURCE UTILIZATION METRICS
# ============================================================================

print("\n=== RESOURCE UTILIZATION METRICS ===")

# Get Spark context information
sc = spark.sparkContext

print("Spark Context Information:")
print(f"Application ID: {sc.applicationId}")
print(f"Application Name: {sc.appName}")
print(f"Master URL: {sc.master}")

# Get executor information (if available)
try:
    # Get executor memory status
    executor_infos = sc._jsc.sc().getExecutorMemoryStatus()
    java_list = executor_infos.toArray()
    print(f"\nExecutor Memory Status ({len(java_list)} executors):")
    for i, info in enumerate(java_list[:5]):  # Show first 5
        host = info._1()
        memory_status = info._2()
        print(f"  Executor {i+1} ({host}): {memory_status}")
    if len(java_list) > 5:
        print(f"  ... and {len(java_list) - 5} more executors")
except Exception as e:
    print(f"Could not retrieve detailed executor information: {e}")
    print("This is normal in some deployment environments")

# Get job and stage information (after performing an action)
print("\nJob and Stage Information (after sample action):")

# Clear any existing job/stage status
sc._jsc.sc().clearJobGroup()

# Perform an action to generate job/stage metrics
start_time = time.time()
count_result = df_processed.count()
action_time = time.time() - start_time

try:
    # Get recent job IDs
    job_ids = sc._jsc.sc().getJobIds()
    if len(job_ids) > 0:
        latest_job_id = job_ids[-1]
        print(f"Latest Job ID: {latest_job_id}")
        
        # Get job information
        job_info = sc._jsc.sc().getJobInfo(latest_job_id)
        if job_info is not None:
            print(f"Job Status: {job_info.status()}")
            print(f"Number of Stages: {job_info.numStages()}")
            print(f"Number of Tasks: {job_info.numTasks()}")
            print(f"Submission Time: {job_info.submissionTime()}")
            print(f"Completion Time: {job_info.completionTime()}")
    else:
        print("No jobs found in scheduler")
except Exception as e:
    print(f"Could not retrieve job information: {e}")

try:
    # Get recent stage IDs
    stage_ids = sc._jsc.sc().getStageIds()
    if len(stage_ids) > 0:
        latest_stage_id = stage_ids[-1]
        print(f"\nLatest Stage ID: {latest_stage_id}")
        
        # Get stage information
        stage_info = sc._jsc.sc().getStageInfo(latest_stage_id)
        if stage_info is not None:
            print(f"Stage Status: {stage_info.status()}")
            print(f"Number of Tasks: {stage_info.numTasks()}")
            print(f"Number of Attempts: {stage_info.numAttempts()}")
            print(f"Submission Time: {stage_info.submissionTime()}")
            print(f"Completion Time: {stage_info.completionTime()}")
            print(f"Failure Reason: {stage_info.failureReason() if stage_info.failureReason() else 'None'}")
    
except Exception as e:
    print(f"Could not retrieve stage information: {e}")

# Record metrics as required
print(f"\n=== REQUIRED METRICS FOR REPORT ===")
print(f"Number of partitions: {df_processed.rdd.getNumPartitions()}")
print(f"Spark configuration: Available via spark.sparkContext.getConf().getAll()")
print(f"Number of jobs: Monitorable via Spark UI Jobs tab")
print(f"Number of stages: Monitorable via Spark UI Stages tab")
print(f"Executor memory: {conf.get('spark.executor.memory', '1g (default)')}")
print(f"Number of cores: {conf.get('spark.executor.cores', '1 (default)')}")